#### Read files from [NCI-ALMANAC](https://discover.nci.nih.gov/rsconnect/cellminercdb/), include:

- Compound 2D structure: ComboCompoundSet.sdf

- Compound chemical names: ComboCompoundNames_small.txt, ComboCompoundNames_all.txt

- Growth Inhibition Data: ComboDrugGrowth_Nov2017.csv

- Documentation: ALMANAC_DataFields.txt / ALMANAC Data Fields.docx

In [43]:
# Get the drug sample numbers and names:

import pandas as pd

# Path to your text file
txt_file_path = "/nfs/turbo/med-kayvan-lab/Projects/DrugCombination/b-DrugCombination/DC_Data/NCI-ALMANAC/ComboCompoundNames_small.txt"

# Read the text file into a DataFrame
df = pd.read_csv(txt_file_path, delimiter='\t')

# Display the DataFrame
print(df.info())
print(df.head())
print(df.tail())

# # Save the DataFrame to a CSV file
# df.to_csv('DrugNSC.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108 entries, 0 to 107
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   #       108 non-null    int64 
 1   Name    108 non-null    object
dtypes: int64(1), object(1)
memory usage: 1.8+ KB
None
     #            Name
0  740    Methotrexate
1  750        Busulfan
2  752     Thioguanine
3  752   6-Thioguanine
4  755  Mercaptopurine
          #                     Name
103  761431              vemurafenib
104  707389        Eribulin mesylate
105  737754  Pazopanib hydrochloride
106  753082              vemurafenib
107  763371              Ruxolitinib


In [49]:
## There are three '#' values assigned to different 'Name's that are the same,
## Align each of the 'Name's with ' / '.
import pandas as pd

# Find the count of each value in the column
count_series = df['#'].value_counts()

# Filter the count series to find the repeated numbers
repeated_numbers = count_series[count_series > 1]

print("Repeated numbers:")
print(repeated_numbers)

# Define a custom function to join the content with " / "
def merge_content(group):
    return ' / '.join(group)

# Group by 'Numbers' and aggregate 'Content' using the custom function
merged_df = df.groupby('#')['Name'].agg(merge_content).reset_index()

print(merged_df)

# Save the DataFrame to a CSV file
merged_df.to_csv('DrugNSC.csv', index=False)

Repeated numbers:
#
49842    2
752      2
24559    2
Name: count, dtype: int64
          #                           Name
0       740                   Methotrexate
1       750                       Busulfan
2       752    Thioguanine / 6-Thioguanine
3       755                 Mercaptopurine
4       762  Mechlorethamine hydrochloride
..      ...                            ...
100  757441                       Axitinib
101  760766                     Vandetanib
102  761431                    vemurafenib
103  761432                    Cabazitaxel
104  763371                    Ruxolitinib

[105 rows x 2 columns]


In [50]:
## There are 2 same 'Name's assigned to two different '#'.
## I don't know which one should be eliminated because, for the 2D structure sdf file, there are no matching '#'s.
import pandas as pd

# Find duplicate values in the "Name" column
duplicates = merged_df[merged_df.duplicated(subset=['Name'], keep=False)]

# Group the duplicate values by the "Name" column and display the '#' values
duplicate_groups = duplicates.groupby('Name')['#'].apply(list)
print(duplicate_groups)

Name
vemurafenib    [753082, 761431]
Name: #, dtype: object


In [1]:
# Read the Compound 2D structure: ComboCompoundSet.sdf file

from rdkit import Chem
from rdkit.Chem import AllChem

# Path to your SDF file
sdf_file_path = "/nfs/turbo/med-kayvan-lab/Projects/DrugCombination/b-DrugCombination/DC_Data/NCI-ALMANAC/ComboCompoundSet.sdf"

# Read the SDF file
suppl = Chem.SDMolSupplier(sdf_file_path)

# Get the number of molecules in the supplier
num_molecules = len(suppl)

print("Number of molecules:", num_molecules)

# Counter for valid molecules
valid_molecules_count = 0

# Iterate over the molecules in the SDF file
for idx, mol in enumerate(suppl, start=1):
    try:
        # Try to sanitize the molecule
        Chem.SanitizeMol(mol)
        # If successful, process the molecule
        if mol is not None:
            # Do something with the molecule
            # For example, you can print its SMILES representation
            print(f"Molecule {idx}: {Chem.MolToSmiles(mol)}")
            valid_molecules_count += 1
    except Exception as e:
        # If an error occurs during sanitization, skip the molecule and print the error
        print(f"Error processing molecule {idx}: {e}")

print("Number of valid molecules:", valid_molecules_count)

Number of molecules: 104
Molecule 1: CN(Cc1cnc2nc(N)nc(N)c2n1)c1ccc(C(=O)N[C@@H](CCC(=O)O)C(=O)O)cc1
Molecule 2: CS(=O)(=O)OCCCCOS(C)(=O)=O
Molecule 3: Nc1nc(=S)c2[nH]cnc2[nH]1
Molecule 4: S=c1nc[nH]c2nc[nH]c12
Molecule 5: CN(CCCl)CCCl.Cl
Molecule 6: O=c1[nH]cnc2[nH]ncc12
Molecule 7: Cc1c2oc3c(C)ccc(C(=O)N[C@@H]4C(=O)N[C@H](C(C)C)C(=O)N5CCC[C@H]5C(=O)N(C)CC(=O)N(C)[C@@H](C(C)C)C(=O)O[C@H]4C)c3nc-2c(C(=O)N[C@@H]2C(=O)N[C@H](C(C)C)C(=O)N3CCC[C@H]3C(=O)N(C)CC(=O)N(C)[C@@H](C(C)C)C(=O)O[C@H]2C)c(N)c1=O
Molecule 8: O=C(O)CCCc1ccc(N(CCCl)CCCl)cc1
Molecule 9: S=P(N1CC1)(N1CC1)N1CC1
Molecule 10: Cl.N[C@@H](Cc1ccc(N(CCCl)CCCl)cc1)C(=O)O
Molecule 11: C1CN1c1nc(N2CC2)nc(N2CC2)n1
Molecule 12: CN(C)c1nc(N(C)C)nc(N(C)C)n1
Molecule 13: CCN(CC)CCCC(C)Nc1c2ccc(Cl)cc2nc2ccc(OC)cc12.Cl
Molecule 14: Cl.NCC(=O)CCC(=O)O
Molecule 15: O=c1[nH]cc(F)c(=O)[nH]1
Molecule 16: CO[C@H](C(=O)[C@@H](O)[C@@H](C)O)[C@@H]1Cc2cc3cc(OC4CC(OC5CC(O)C(O)C(C)O5)C(O)C(C)O4)c(C)c(O)c3c(O)c2C(=O)[C@H]1OC1CC(OC2CC(OC3CC(C)(O)C(O)C

[11:00:19] Explicit valence for atom # 0 Cl, 2, is greater than permitted
[11:00:19] ERROR: Could not sanitize molecule ending on line 6200
[11:00:19] ERROR: Explicit valence for atom # 0 Cl, 2, is greater than permitted
[11:00:19] Explicit valence for atom # 3 O, 3, is greater than permitted
[11:00:19] ERROR: Could not sanitize molecule ending on line 8652
[11:00:19] ERROR: Explicit valence for atom # 3 O, 3, is greater than permitted


In [2]:
# Read the Growth Inhibition Data: ComboDrugGrowth_Nov2017.csv file

import pandas as pd

# Path to your CSV file
csv_file_path = "/nfs/turbo/med-kayvan-lab/Projects/DrugCombination/b-DrugCombination/DC_Data/NCI-ALMANAC/ComboDrugGrowth_Nov2017.csv"

# Read the CSV file
df = pd.read_csv(csv_file_path)

# Display the first few rows of the DataFrame to verify the data has been read correctly
print(df.info())
print(df.head())
print(df.tail())

/tmp/ipykernel_3073708/357718203.py:9: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_file_path)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3686475 entries, 0 to 3686474
Data columns (total 29 columns):
 #   Column             Dtype  
---  ------             -----  
 0   COMBODRUGSEQ       int64  
 1   SCREENER           object 
 2   STUDY              object 
 3   TESTDATE           object 
 4   PLATE              object 
 5   PANELNBR           int64  
 6   CELLNBR            int64  
 7   PREFIX1            object 
 8   NSC1               int64  
 9   SAMPLE1            int64  
 10  CONCINDEX1         int64  
 11  CONC1              float64
 12  CONCUNIT1          object 
 13  PREFIX2            object 
 14  NSC2               float64
 15  SAMPLE2            float64
 16  CONCINDEX2         int64  
 17  CONC2              float64
 18  CONCUNIT2          object 
 19  PERCENTGROWTH      float64
 20  PERCENTGROWTHNOTZ  float64
 21  TESTVALUE          float64
 22  CONTROLVALUE       float64
 23  TZVALUE            float64
 24  EXPECTEDGROWTH     float64
 25  SCORE             